# Stage 07 - Boltz-2 cofolding of the selected designs

Predicts each selected binder sequence in complex with three targets, which are
the project's three ranking criteria in order:

| condition | chains | question |
|---|---|---|
| `g12d` | HLA alpha1/alpha2 + `VVVGADGVGK` + binder | does it bind the target? |
| `wt`   | HLA alpha1/alpha2 + `VVVGAGGVGK` + binder | does it bind without Asp6? |
| `apo`  | HLA alpha1/alpha2 + binder | does it bind the empty groove? |

The g12d/wt pair differs at peptide position 6 alone, so the comparison isolates
the mutation. g12d against apo isolates peptide dependence.

**Why this stage exists at all.** Stage 05 produces backbones with no side chains,
and every contact-based method - PRODIGY included - needs full-atom side chains on
both partners. Cofolding supplies them from sequence, which is the only route to
them without a packer. It also returns `pair_chains_iptm`, the per-chain-pair
interface confidence that binder campaigns rank on.

**Manual steps**

1. **Runtime -> Change runtime type -> Runtime version 2026.04** and a GPU.
   Mandatory: on the default Python 3.13 runtime, `pip install boltz` resolves to
   boltz 2.0.3 instead of 2.2.1. Verified by resolving the dependency for each
   version on 9 Nov 2026.
2. Run cells 1-5. Cell 4 computes the HLA MSA **once**; cell 5 runs the
   predictions and is re-runnable, skipping whatever is already on Drive.
3. On the Mac: `make cofolding` to ingest and score.

Inputs come from `scripts/07a_stage_cofolding.py`. Nothing is embedded here.

In [ ]:
#@title 1. Parameters
DRIVE_FOLDER = "kras_pmhc_stage07"   # under My Drive; set by 07a_stage_cofolding.py

# Pinned: 2.2.1 is what Python 3.12 resolves to, and the version this stage was
# planned against. An unpinned install on a 3.13 runtime silently gives 2.0.3.
BOLTZ_VERSION = "2.2.1"

# One diffusion sample per prediction. 42 predictions have to fit in an evening,
# and the quantities this stage feeds - interface confidence and a full-atom
# complex for contact counting - are read from the top-ranked sample anyway.
DIFFUSION_SAMPLES = 1
RECYCLING_STEPS = 3

# The token scripts/07a_stage_cofolding.py writes where the HLA MSA path goes.
MSA_TOKEN = "__HLA_MSA__"

BASE = "/content/stage07"

print(f"boltz {BOLTZ_VERSION}, {DIFFUSION_SAMPLES} sample(s), "
      f"{RECYCLING_STEPS} recycling steps")

In [ ]:
#@title 2. Environment (~5 min)
import os
import subprocess
import sys
from pathlib import Path


def sh(command, check=True):
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"failed ({result.returncode}): {command}")
    return result


print(f"python {sys.version.split()[0]}")
if sys.version_info[:2] >= (3, 13):
    raise RuntimeError(
        f"this runtime is Python {sys.version.split()[0]}, where pip resolves "
        "boltz to 2.0.3. Set Runtime version to 2026.04 (Python 3.12) and "
        "re-run - do not continue, the results would not be the pinned model."
    )

Path(BASE).mkdir(parents=True, exist_ok=True)
os.chdir(BASE)

if sh("which boltz", check=False).returncode != 0:
    print(f"installing boltz {BOLTZ_VERSION}...")
    sh(f"pip install -q boltz=={BOLTZ_VERSION}")

installed = sh("pip show boltz").stdout
version = next(
    (line.split(":", 1)[1].strip() for line in installed.splitlines()
     if line.startswith("Version:")),
    "unknown",
)
print(f"boltz {version}")
if version != BOLTZ_VERSION:
    raise RuntimeError(f"wanted boltz {BOLTZ_VERSION}, got {version}")

print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv").stdout.strip())

In [ ]:
#@title 3. Mount Drive and verify the staged inputs
import csv
import hashlib

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
INPUTS = DRIVE / "inputs"
OUTPUTS = DRIVE / "predictions"
OUTPUTS.mkdir(parents=True, exist_ok=True)

with open(DRIVE / "JOBS.tsv") as handle:
    JOBS = list(csv.DictReader(handle, delimiter="\t"))

# Every input is checksummed at staging. A half-synced YAML would predict a
# truncated sequence and report confidence for it, which is worse than failing.
for job in JOBS:
    text = (INPUTS / job["yaml_file"]).read_text()
    digest = hashlib.sha256(text.encode()).hexdigest()
    if digest != job["sha256"]:
        raise ValueError(f"checksum mismatch for {job['yaml_file']}")

conditions = {}
for job in JOBS:
    conditions[job["condition"]] = conditions.get(job["condition"], 0) + 1
print(f"{len(JOBS)} jobs verified against JOBS.tsv: {conditions}")
print(f"designs: {len({j['backbone_id'] for j in JOBS})}")
print(f"outputs -> {OUTPUTS}")

In [ ]:
#@title 4. Compute the HLA MSA once
# The HLA groove sequence is identical in all 42 predictions, so its alignment is
# computed once and reused. Querying the public MSA server 42 times for the same
# sequence would be slower and discourteous. The binder is de novo and the
# peptide is a 10-mer, so both run single-sequence - that is in the staged YAMLs.
MSA_PATH = DRIVE / "hla_groove.a3m"

if MSA_PATH.exists():
    print(f"reusing {MSA_PATH.name} ({MSA_PATH.stat().st_size} bytes)")
else:
    hla = next(
        line.split(": ", 1)[1].strip()
        for line in (INPUTS / JOBS[0]["yaml_file"]).read_text().splitlines()
        if line.strip().startswith("sequence:")
    )
    print(f"querying the MSA server for {len(hla)} residues...")

    probe_dir = Path(BASE) / "msa_probe"
    probe_dir.mkdir(exist_ok=True)
    probe = probe_dir / "hla_only.yaml"
    # No msa: field, so --use_msa_server supplies one.
    probe.write_text(
        "version: 1\nsequences:\n  - protein:\n      id: A\n"
        f"      sequence: {hla}\n"
    )
    sh(f"boltz predict {probe} --out_dir {probe_dir} --use_msa_server "
       f"--diffusion_samples 1 --recycling_steps 1 --output_format mmcif")

    found = sorted(probe_dir.rglob("*.a3m"), key=lambda p: p.stat().st_size)
    if not found:
        raise RuntimeError(
            "no .a3m produced. Fall back to running cell 5 with "
            "USE_MSA_SERVER_PER_JOB = True, which queries per prediction."
        )
    MSA_PATH.write_bytes(found[-1].read_bytes())
    print(f"wrote {MSA_PATH} from {found[-1].name} "
          f"({MSA_PATH.stat().st_size} bytes)")

In [ ]:
#@title 5. Predict - re-runnable, skips what is already on Drive
import json
import shutil
import time

#: Set True only if cell 4 could not produce an alignment.
USE_MSA_SERVER_PER_JOB = False

WORK = Path(BASE) / "work"
WORK.mkdir(exist_ok=True)


def done(job):
    """A job counts as done when both artefacts are on Drive."""
    out = OUTPUTS / job["job_id"]
    return (out / "model.cif").exists() and (out / "confidence.json").exists()


def predict(job):
    text = (INPUTS / job["yaml_file"]).read_text()
    if USE_MSA_SERVER_PER_JOB:
        text = "\n".join(
            line for line in text.splitlines() if MSA_TOKEN not in line
        ) + "\n"
    else:
        text = text.replace(MSA_TOKEN, str(MSA_PATH))

    local = WORK / job["yaml_file"]
    local.write_text(text)
    run_dir = WORK / job["job_id"]

    flags = [
        f"--out_dir {run_dir}",
        f"--diffusion_samples {DIFFUSION_SAMPLES}",
        f"--recycling_steps {RECYCLING_STEPS}",
        "--output_format mmcif",
        "--override",
    ]
    if USE_MSA_SERVER_PER_JOB:
        flags.append("--use_msa_server")
    command = f"boltz predict {local} " + " ".join(flags)

    started = time.time()
    result = sh(command, check=False)
    elapsed = time.time() - started
    if result.returncode != 0:
        print(result.stdout[-1500:])
        print(result.stderr[-1500:])
        raise RuntimeError(f"{job['job_id']} failed ({result.returncode})")

    # Copy out by pattern rather than by assumed path: the layout under
    # boltz_results_* has changed between versions, and a wrong guess here would
    # look like a prediction failure.
    cif = sorted(run_dir.rglob("*_model_0.cif")) or sorted(run_dir.rglob("*.cif"))
    conf = sorted(run_dir.rglob("confidence_*_model_0.json")) or \
        sorted(run_dir.rglob("confidence*.json"))
    if not cif or not conf:
        raise RuntimeError(
            f"{job['job_id']}: no model/confidence found under {run_dir}"
        )

    out = OUTPUTS / job["job_id"]
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cif[0], out / "model.cif")
    shutil.copy2(conf[0], out / "confidence.json")
    shutil.rmtree(run_dir, ignore_errors=True)
    return elapsed, json.loads((out / "confidence.json").read_text())


pending = [job for job in JOBS if not done(job)]
print(f"{len(JOBS) - len(pending)} already on Drive, {len(pending)} to run\n")

failures = []
for index, job in enumerate(pending, start=1):
    print(f"[{index}/{len(pending)}] {job['job_id']}")
    try:
        elapsed, confidence = predict(job)
    except RuntimeError as error:
        print(f"  FAILED: {error}")
        failures.append(job["job_id"])
        continue
    iptm = confidence.get("iptm")
    plddt = confidence.get("complex_plddt")
    print(f"  {elapsed:.0f}s  iptm {iptm}  complex_plddt {plddt}")

print(f"\n{sum(1 for job in JOBS if done(job))}/{len(JOBS)} complete")
if failures:
    print(f"failed: {failures}")
print("\nOn the Mac:  make cofolding")